# Notebook 06 — Amazon-like product-aware QA generativo senza API

Questo notebook costruisce un primo sistema di **Question Answering generativo sui prodotti** per il dataset **Amazon-like / test_dataset2**.

La parola *generativo* qui va interpretata in modo prudente: il notebook non chiama API esterne e non usa un grande modello linguistico online. Genera una risposta nuova combinando:

1. metadati del prodotto;
2. coppie domanda-risposta già presenti nel dataset;
3. recensioni testuali salvate dal Notebook 01;
4. una fase di recupero delle evidenze più rilevanti;
5. una fase di composizione controllata della risposta.

Quindi il sistema non si limita a restituire una QA già esistente, ma produce una risposta leggibile basata sulle evidenze trovate. Se le evidenze sono deboli o contraddittorie, il sistema lo dichiara invece di inventare.


## 1. Collegamento con il paper

Nel paper *Information Discovery in E-commerce*, il QA e-commerce viene descritto come un sistema che aiuta gli utenti a ottenere risposte dirette e concise su domande relative a un prodotto. Il paper distingue tra sistemi retrieval-based, che cercano risposte nei dati, e sistemi generation-based, che producono una risposta a partire dalle informazioni disponibili.

La parte più vicina a questo notebook è la **generative product-aware QA**: dato un prodotto, una domanda, recensioni e attributi, il sistema dovrebbe leggere le recensioni, individuare le parti rilevanti, combinarle con gli attributi e produrre una risposta coerente.

Per restare realistici per una tesi triennale, qui non implementiamo un modello neurale complesso come PAAG o RAHGNN. Implementiamo invece una versione leggera e spiegabile:

```text
domanda utente
→ recupero evidenze da QA, recensioni e metadati
→ stima semplice del tipo di domanda e del supporto positivo/negativo
→ generazione controllata di una risposta
→ visualizzazione delle evidenze usate
```

Questa scelta è meno potente di un modello generativo profondo, ma è più verificabile, non usa API esterne e permette di mostrare chiaramente su quali dati si basa ogni risposta.


## 2. Setup

In [1]:
#Installiamo solo librerie leggere usate dal notebook
#Non installiamo modelli generativi esterni e non chiamiamo API
#Esegui questa cella solo se manca qualche pacchetto

%pip install -q pandas pyarrow numpy scipy scikit-learn joblib tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Importiamo librerie per dati, testo, vettori sparsi e salvataggio artefatti
#Il notebook è autonomo rispetto ai notebook precedenti
#Gli input vengono letti dalla cartella data/processed della root del progetto

from pathlib import Path
import json
import re
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import joblib
import pyarrow.parquet as pq

from IPython.display import display
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Percorsi e configurazione

Il notebook può stare dentro `notebooks/test_dataset2`, ma **deve leggere i dati dalla cartella generale della repo**, non da una cartella `data` locale dentro `test_dataset2`.

In questa versione la root è impostata esplicitamente:

```python
PROJECT_ROOT=Path(r"C:\\Users\\arsid\\Desktop\\Tesi-ecommerce")
```

I file Amazon-like vengono letti dalle stesse due cartelle usate dagli altri notebook Amazon-like:

```text
Tesi-ecommerce/data/processed/test_dataset2_intersection/
Tesi-ecommerce/data/processed/test_dataset2_recommender/
```

Gli artefatti prodotti da questo notebook vengono salvati sotto `test_dataset2_recommender/notebook06_generative_qa`.


In [3]:
#Definiamo parametri principali e percorsi del dataset Amazon-like
#La root del progetto è impostata manualmente come richiesto
#I dati vengono letti dalle cartelle test_dataset2 usate dagli altri notebook

DATASET_NAME="test_dataset2"
RANDOM_STATE=42

MAX_TOTAL_REVIEW_EVIDENCE=250000
MAX_REVIEWS_PER_PRODUCT=40
MIN_REVIEW_TEXT_CHARS=25
MIN_EVIDENCE_SCORE=0.08
TOP_EVIDENCE=8

CURRENT_DIR=Path.cwd().resolve()
PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")
PROCESSED_DIR=PROJECT_ROOT/"data"/"processed"
INTERSECTION_DIR=PROCESSED_DIR/"test_dataset2_intersection"
RECOMMENDER_DIR=PROCESSED_DIR/"test_dataset2_recommender"

#Restituiamo il primo percorso esistente tra più alternative
#Se nessuno esiste, restituiamo il primo per produrre un errore leggibile più avanti
#Questo aiuta a supportare sia output del notebook 01 sia output del notebook 03

def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return paths[0]

ITEMS_PATH=first_existing([RECOMMENDER_DIR/"items.parquet",INTERSECTION_DIR/"items.parquet"])
QA_PATH=first_existing([RECOMMENDER_DIR/"qa.parquet",INTERSECTION_DIR/"qa.parquet"])
RATINGS_PATH=first_existing([INTERSECTION_DIR/"ratings.parquet",RECOMMENDER_DIR/"ratings.parquet"])

NOTEBOOK06_DIR=RECOMMENDER_DIR/"notebook06_generative_qa"
NOTEBOOK06_DIR.mkdir(parents=True,exist_ok=True)

EVIDENCE_CORPUS_PATH=NOTEBOOK06_DIR/"qa_evidence_corpus.parquet"
VECTORIZER_PATH=NOTEBOOK06_DIR/"qa_evidence_tfidf_vectorizer.joblib"
MATRIX_PATH=NOTEBOOK06_DIR/"qa_evidence_tfidf_matrix.npz"
SUMMARY_PATH=NOTEBOOK06_DIR/"notebook06_generative_qa_summary.json"

print("Current dir:",CURRENT_DIR)
print("Project root:",PROJECT_ROOT)
print("Processed dir:",PROCESSED_DIR)
print("Intersection dir:",INTERSECTION_DIR,"| exists=",INTERSECTION_DIR.exists())
print("Recommender dir:",RECOMMENDER_DIR,"| exists=",RECOMMENDER_DIR.exists())
print("Notebook06 dir:",NOTEBOOK06_DIR)
print("Items path:",ITEMS_PATH,"| exists=",ITEMS_PATH.exists())
print("QA path:",QA_PATH,"| exists=",QA_PATH.exists())
print("Ratings path:",RATINGS_PATH,"| exists=",RATINGS_PATH.exists())


Current dir: C:\Users\arsid\Desktop\Tesi-ecommerce\notebooks\test_dataset2
Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Processed dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed
Intersection dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_intersection | exists= True
Recommender dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_recommender | exists= True
Notebook06 dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_recommender\notebook06_generative_qa
Items path: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_recommender\items.parquet | exists= True
QA path: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_recommender\qa.parquet | exists= True
Ratings path: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_intersection\ratings.parquet | exists= True


## 4. Caricamento prodotti e QA

In [4]:
#Carichiamo prodotti e coppie domanda-risposta
#Normalizziamo parent_asin come stringa
#Useremo questi dati come attributi e conoscenza esplicita del prodotto

if not ITEMS_PATH.exists():
    raise FileNotFoundError("File items.parquet non trovato: "+str(ITEMS_PATH))

if not QA_PATH.exists():
    raise FileNotFoundError("File qa.parquet non trovato: "+str(QA_PATH))

items_df=pd.read_parquet(ITEMS_PATH)
qa_df=pd.read_parquet(QA_PATH)

items_df=items_df.copy()
qa_df=qa_df.copy()
items_df["parent_asin"]=items_df["parent_asin"].astype(str)
qa_df["parent_asin"]=qa_df["parent_asin"].astype(str)

items_df=items_df.drop_duplicates("parent_asin").reset_index(drop=True)
valid_products=set(items_df["parent_asin"])
qa_df=qa_df[qa_df["parent_asin"].isin(valid_products)].reset_index(drop=True)

print("items_df:",items_df.shape)
print("qa_df:",qa_df.shape)
print("Prodotti validi:",len(valid_products))
print("Prodotti con QA:",qa_df["parent_asin"].nunique())

display(items_df.head(3))
display(qa_df.head(3))

items_df: (19000, 17)
qa_df: (166191, 13)
Prodotti validi: 19000
Prodotti con QA: 19000


,parent_asin,source_category,hf_category,qa_slug,category_group,title,category,avg_rating,num_ratings,price,store,features_json,description_json,categories_json,details_json,image_url,item_idx
0,B00BGFTSGU,Electronics,Electronics,Electronics,tech,GGS Swivi HD DSLR LCD Universal Foldable Viewf...,Camera & Photo,3.7,52,149.00,Swivi,[],[],"[""Electronics"", ""Camera & Photo"", ""Accessories...","""{\""Product Dimensions\"": \""3.8 x 4.3 x 6.9 in...",https://m.media-amazon.com/images/I/41BUv5PjMD...,15043
1,B00AZOIG9I,Electronics,Electronics,Electronics,tech,BM Premium 2-Pack of NP-85 Batteries and Charg...,Camera & Photo,4.7,345,22.99,BM Premium,"[""NP85 Li-ion Battery for FujiFilm FinePix SL2...","[""Bring your digital camera back to life with ...","[""Electronics"", ""Camera & Photo"", ""Accessories...","""{\""Package Dimensions\"": \""6.46 x 3.94 x 1.85...",https://m.media-amazon.com/images/I/71fbU-87Tx...,14636
2,B00F361KLO,Electronics,Electronics,Electronics,tech,ZAGG InvisibleShield HD – EZ Apply Film Screen...,Computers,4.1,158,NaN,InvisibleShield,"[""100% Clear: Independent light transmission t...","[""Exceptionally clear, unbelievably thin, and ...","[""Electronics"", ""Computers & Accessories"", ""Ta...","""{\""Standing screen display size\"": \""9.7 Inch...",https://m.media-amazon.com/images/I/71XY4kKwcX...,17360


,qa_id,parent_asin,source_category,hf_category,qa_slug,category_group,question,answer,questionType,answerType,answerTime,unixTime,item_idx
0,10,0972683275,Electronics,Electronics,Electronics,tech,how far out does the arm extend?,18 inches on our TV.,open-ended,NaN,"Dec 3, 2013",1.386058e+09,2
1,11,0972683275,Electronics,Electronics,Electronics,tech,"can you mount a sound bar using the external ""...","Yes, I can but need sound bar mount.",yes/no,Y,"Oct 27, 2013",1.382857e+09,2
2,12,0972683275,Electronics,Electronics,Electronics,tech,My TV has a VESA 200x200 bolt pattern and I ne...,200mm x200mm. Yes Fully extended from wall to ...,yes/no,Y,"Nov 27, 2012",1.354003e+09,2


## 5. Controllo testo recensioni

Per generare risposte basate sulle recensioni serve che `ratings.parquet` contenga almeno `review_text`. Se manca, significa che nel Notebook 01 non è stato salvato il testo delle recensioni oppure non è stato forzato il rebuild del file.


In [5]:
#Controlliamo lo schema del file ratings.parquet
#Il QA generativo ha bisogno della colonna review_text
#Se non esiste, bisogna rieseguire il Notebook 01 con SAVE_TEXT_OFREVIEW=True

if not RATINGS_PATH.exists():
    raise FileNotFoundError("File ratings.parquet non trovato: "+str(RATINGS_PATH))

ratings_schema=pq.ParquetFile(RATINGS_PATH).schema_arrow.names
print("Colonne disponibili in ratings.parquet:")
print(ratings_schema)

if "review_text" not in ratings_schema:
    raise ValueError("Manca review_text in ratings.parquet. Riesegeui il Notebook 01 con SAVE_TEXT_OFREVIEW=True e FORCE_REBUILD_RATINGS=True")

Colonne disponibili in ratings.parquet:
['user_id', 'parent_asin', 'source_category', 'hf_category', 'qa_slug', 'category_group', 'asin', 'rating', 'timestamp', 'verified_purchase', 'review_title', 'review_text']


## 6. Funzioni di pulizia testo e attributi

In [6]:
#Definiamo funzioni di utilità per trasformare valori complessi in testo
#Le colonne dei metadati possono contenere liste, dizionari o valori nulli
#Queste funzioni rendono i testi più regolari per indicizzazione e risposta

def is_missing(value):
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except Exception:
        return False

def flatten_value(value):
    if is_missing(value):
        return ""
    if isinstance(value,(list,tuple,set)):
        parts=[flatten_value(v) for v in value]
        return " ".join([p for p in parts if p])
    if isinstance(value,dict):
        parts=[]
        for key,val in value.items():
            key_text=flatten_value(key)
            val_text=flatten_value(val)
            if key_text and val_text:
                parts.append(key_text+": "+val_text)
            elif val_text:
                parts.append(val_text)
        return " ".join(parts)
    return str(value)

def clean_text(text):
    text=flatten_value(text)
    text=re.sub(r"\s+"," ",text)
    text=text.replace("\n"," ").replace("\r"," ")
    return text.strip()

def shorten_text(text,max_chars=500):
    text=clean_text(text)
    if len(text)<=max_chars:
        return text
    return text[:max_chars].rsplit(" ",1)[0]+"..."

def split_sentences(text):
    text=clean_text(text)
    if not text:
        return []
    sentences=re.split(r"(?<=[.!?])\s+",text)
    return [s.strip() for s in sentences if len(s.strip())>=10]

def product_title(parent_asin):
    row=items_by_asin.get(str(parent_asin))
    if row is None:
        return str(parent_asin)
    for col in ["title","product_title","name"]:
        if col in row and clean_text(row[col]):
            return clean_text(row[col])
    return str(parent_asin)

In [7]:
#Creiamo un dizionario veloce parent_asin -> metadati prodotto
#Questo serve sia per risposte dirette sia per costruire evidenze metadata
#Le righe sono convertite in dizionari per semplificare l'accesso

items_by_asin={row["parent_asin"]:row for row in items_df.to_dict(orient="records")}

metadata_text_columns=[
    "title",
    "main_category",
    "categories",
    "description",
    "features",
    "details",
    "store",
    "brand",
    "price",
    "average_rating",
    "rating_number",
]

available_metadata_text_columns=[col for col in metadata_text_columns if col in items_df.columns]
print("Colonne metadata usate:",available_metadata_text_columns)

Colonne metadata usate: ['title', 'store', 'price']


## 7. Caricamento campione recensioni testuali

Per non creare un indice enorme, il notebook usa al massimo `MAX_TOTAL_REVIEW_EVIDENCE` recensioni e al massimo `MAX_REVIEWS_PER_PRODUCT` recensioni per prodotto.

Questa è una scelta pratica: per un prototipo QA non serve indicizzare tutte le recensioni se il file è molto grande. Conviene avere abbastanza evidenze per prodotto, ma mantenere tempi e memoria sotto controllo.


In [8]:
#Carichiamo recensioni testuali leggendo il parquet a batch
#Questo evita di caricare subito in memoria tutto il file ratings.parquet
#Manteniamo solo recensioni associate ai prodotti del dataset Amazon-like/test_dataset2

def load_review_rows(path,valid_products):
    parquet_file=pq.ParquetFile(path)
    available_cols=parquet_file.schema_arrow.names
    wanted_cols=["parent_asin","rating","timestamp","verified_purchase","review_title","review_text"]
    columns=[col for col in wanted_cols if col in available_cols]
    rows=[]
    product_counts=defaultdict(int)
    total_kept=0

    for batch in tqdm(parquet_file.iter_batches(batch_size=100000,columns=columns),desc="Caricamento recensioni"):
        batch_df=batch.to_pandas()
        if "parent_asin" not in batch_df.columns or "review_text" not in batch_df.columns:
            continue
        batch_df["parent_asin"]=batch_df["parent_asin"].astype(str)
        batch_df=batch_df[batch_df["parent_asin"].isin(valid_products)].copy()
        batch_df["review_text"]=batch_df["review_text"].fillna("").astype(str)
        batch_df=batch_df[batch_df["review_text"].str.len()>=MIN_REVIEW_TEXT_CHARS]

        for row in batch_df.itertuples(index=False):
            parent_asin=str(getattr(row,"parent_asin"))
            if product_counts[parent_asin]>=MAX_REVIEWS_PER_PRODUCT:
                continue
            row_dict={col:getattr(row,col) for col in batch_df.columns}
            rows.append(row_dict)
            product_counts[parent_asin]+=1
            total_kept+=1
            if total_kept>=MAX_TOTAL_REVIEW_EVIDENCE:
                return pd.DataFrame(rows)

    return pd.DataFrame(rows)

reviews_df=load_review_rows(RATINGS_PATH,valid_products)

print("Recensioni testuali caricate:",len(reviews_df))
print("Prodotti con almeno una review caricata:",reviews_df["parent_asin"].nunique() if not reviews_df.empty else 0)
display(reviews_df.head(3))

Caricamento recensioni: 5it [00:03,  1.35it/s]

Recensioni testuali caricate: 250000
Prodotti con almeno una review caricata: 13644


,parent_asin,rating,timestamp,verified_purchase,review_title,review_text
0,B0047T79VS,3.0,1344406083000,True,marginal sound quality,this really deserves 3 stars. its not terribl...
1,B001FWYGJS,4.0,1427592277000,True,good and cheap,Had for about a month now. Still running stron...
2,B00BHC7ZTE,5.0,1432342265000,True,Great Product,Excellent sound. Easy to install.


## 8. Costruzione corpus di evidenze

Il corpus contiene tre tipi di evidenza:

- `metadata`: attributi del prodotto;
- `qa`: domande e risposte storiche;
- `review`: recensioni testuali degli utenti.

La risposta finale verrà generata solo a partire da queste evidenze.


In [9]:
#Creiamo le evidenze metadata
#Ogni prodotto genera una riga testuale con titolo, categoria, descrizione e attributi disponibili
#Questa sorgente aiuta soprattutto con domande su categoria, prezzo, rating e caratteristiche

metadata_rows=[]

for row in items_df.to_dict(orient="records"):
    parent_asin=str(row["parent_asin"])
    parts=[]
    for col in available_metadata_text_columns:
        value=clean_text(row.get(col,""))
        if value:
            parts.append(col+": "+value)
    text=". ".join(parts)
    if text:
        metadata_rows.append({
            "parent_asin":parent_asin,
            "evidence_type":"metadata",
            "text":text,
            "question":"",
            "answer":"",
            "rating":np.nan,
            "timestamp":np.nan,
            "verified_purchase":np.nan,
        })

metadata_evidence_df=pd.DataFrame(metadata_rows)
print("Evidenze metadata:",len(metadata_evidence_df))
display(metadata_evidence_df.head(3))

Evidenze metadata: 19000


,parent_asin,evidence_type,text,question,answer,rating,timestamp,verified_purchase
0,B00BGFTSGU,metadata,title: GGS Swivi HD DSLR LCD Universal Foldabl...,,,NaN,NaN,NaN
1,B00AZOIG9I,metadata,title: BM Premium 2-Pack of NP-85 Batteries an...,,,NaN,NaN,NaN
2,B00F361KLO,metadata,title: ZAGG InvisibleShield HD – EZ Apply Film...,,,NaN,NaN,NaN


In [10]:
#Creiamo le evidenze QA storiche
#La parte retrieval-based resta utile quando esiste già una domanda simile
#Il testo indicizzato unisce domanda e risposta per migliorare il recupero

qa_rows=[]

question_col="question"
answer_col="answer"

for row in qa_df.to_dict(orient="records"):
    question=clean_text(row.get(question_col,""))
    answer=clean_text(row.get(answer_col,""))
    if not question and not answer:
        continue
    qa_rows.append({
        "parent_asin":str(row["parent_asin"]),
        "evidence_type":"qa",
        "text":("Question: "+question+" Answer: "+answer).strip(),
        "question":question,
        "answer":answer,
        "rating":np.nan,
        "timestamp":np.nan,
        "verified_purchase":np.nan,
    })

qa_evidence_df=pd.DataFrame(qa_rows)
print("Evidenze QA:",len(qa_evidence_df))
display(qa_evidence_df.head(3))

Evidenze QA: 166191


,parent_asin,evidence_type,text,question,answer,rating,timestamp,verified_purchase
0,0972683275,qa,Question: how far out does the arm extend? Ans...,how far out does the arm extend?,18 inches on our TV.,NaN,NaN,NaN
1,0972683275,qa,Question: can you mount a sound bar using the ...,"can you mount a sound bar using the external ""...","Yes, I can but need sound bar mount.",NaN,NaN,NaN
2,0972683275,qa,Question: My TV has a VESA 200x200 bolt patter...,My TV has a VESA 200x200 bolt pattern and I ne...,200mm x200mm. Yes Fully extended from wall to ...,NaN,NaN,NaN


In [11]:
#Creiamo le evidenze review
#Ogni recensione diventa una possibile evidenza testuale
#Il rating viene conservato perché aiuta a generare risposte su qualità e opinioni

review_rows=[]

for row in reviews_df.to_dict(orient="records"):
    title=clean_text(row.get("review_title",""))
    body=clean_text(row.get("review_text",""))
    if not body:
        continue
    text=("Review title: "+title+". Review text: "+body).strip() if title else body
    review_rows.append({
        "parent_asin":str(row.get("parent_asin","")),
        "evidence_type":"review",
        "text":text,
        "question":"",
        "answer":"",
        "rating":row.get("rating",np.nan),
        "timestamp":row.get("timestamp",np.nan),
        "verified_purchase":row.get("verified_purchase",np.nan),
    })

review_evidence_df=pd.DataFrame(review_rows)
print("Evidenze review:",len(review_evidence_df))
display(review_evidence_df.head(3))

Evidenze review: 250000


,parent_asin,evidence_type,text,question,answer,rating,timestamp,verified_purchase
0,B0047T79VS,review,Review title: marginal sound quality. Review t...,,,3.0,1344406083000,True
1,B001FWYGJS,review,Review title: good and cheap. Review text: Had...,,,4.0,1427592277000,True
2,B00BHC7ZTE,review,Review title: Great Product. Review text: Exce...,,,5.0,1432342265000,True


In [12]:
#Uniamo metadata, QA e review in un unico corpus
#Aggiungiamo un identificatore numerico stabile per ogni evidenza
#Salviamo il corpus per poterlo riusare senza ricostruirlo manualmente

evidence_corpus_df=pd.concat([metadata_evidence_df,qa_evidence_df,review_evidence_df],ignore_index=True)
evidence_corpus_df=evidence_corpus_df.dropna(subset=["parent_asin","text"]).copy()
evidence_corpus_df["parent_asin"]=evidence_corpus_df["parent_asin"].astype(str)
evidence_corpus_df["text"]=evidence_corpus_df["text"].fillna("").astype(str)
evidence_corpus_df=evidence_corpus_df[evidence_corpus_df["text"].str.len()>=10].reset_index(drop=True)
evidence_corpus_df.insert(0,"evidence_id",np.arange(len(evidence_corpus_df),dtype=np.int64))

evidence_corpus_df.to_parquet(EVIDENCE_CORPUS_PATH,index=False)

print("Corpus evidenze:",evidence_corpus_df.shape)
print("Prodotti nel corpus:",evidence_corpus_df["parent_asin"].nunique())
print("Tipi evidenza:")
display(evidence_corpus_df["evidence_type"].value_counts().to_frame("count"))

Corpus evidenze: (435191, 9)
Prodotti nel corpus: 19000
Tipi evidenza:


,count
evidence_type,
review,250000
qa,166191
metadata,19000


## 9. Indicizzazione TF-IDF delle evidenze

La generazione della risposta è preceduta da una fase di recupero: la domanda viene confrontata con QA, recensioni e metadati del prodotto.

Qui usiamo TF-IDF perché è semplice, locale, riproducibile e spiegabile. Non è il metodo più avanzato possibile, ma è adatto a un prototipo di tesi.


In [13]:
#Costruiamo la matrice TF-IDF del corpus evidenze
#Usiamo parole e bigrammi per migliorare il matching su frasi brevi
#Normalizziamo la matrice per usare il prodotto scalare come similarità coseno

vectorizer=TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),
    min_df=2,
    max_df=0.90,
    max_features=120000,
)

evidence_matrix=vectorizer.fit_transform(evidence_corpus_df["text"].tolist())
evidence_matrix=normalize(evidence_matrix,axis=1)

joblib.dump(vectorizer,VECTORIZER_PATH)
save_npz(MATRIX_PATH,evidence_matrix)

product_to_indices={}
for parent_asin,group in evidence_corpus_df.groupby("parent_asin"):
    product_to_indices[parent_asin]=group.index.to_numpy(dtype=np.int64)

print("Matrice evidenze:",evidence_matrix.shape)
print("Valori non nulli:",evidence_matrix.nnz)
print("Prodotti indicizzati:",len(product_to_indices))

Matrice evidenze: (435191, 120000)
Valori non nulli: 18959129
Prodotti indicizzati: 19000


## 10. Recupero delle evidenze per una domanda

In [14]:
#Recuperiamo le evidenze più simili alla domanda per uno specifico prodotto
#La ricerca resta product-aware perché confronta solo evidenze dello stesso parent_asin
#Se il prodotto non ha evidenze, la funzione restituisce una tabella vuota

def retrieve_evidence(parent_asin,question,top_k=TOP_EVIDENCE):
    parent_asin=str(parent_asin)
    question=clean_text(question)
    if parent_asin not in product_to_indices:
        return pd.DataFrame()

    indices=product_to_indices[parent_asin]
    if len(indices)==0:
        return pd.DataFrame()

    query_vector=vectorizer.transform([question])
    query_vector=normalize(query_vector,axis=1)
    scores=evidence_matrix[indices].dot(query_vector.T).toarray().ravel()
    order=np.argsort(-scores)[:top_k]
    selected_indices=indices[order]

    result_df=evidence_corpus_df.iloc[selected_indices].copy()
    result_df["score"]=scores[order]
    result_df=result_df[result_df["score"]>0].reset_index(drop=True)
    return result_df

def best_sentence_for_question(text,question):
    sentences=split_sentences(text)
    if not sentences:
        return shorten_text(text,300)
    question_terms=set(re.findall(r"[a-zA-Z0-9]+",question.lower()))
    best_sentence=sentences[0]
    best_score=-1
    for sentence in sentences:
        sentence_terms=set(re.findall(r"[a-zA-Z0-9]+",sentence.lower()))
        overlap=len(question_terms & sentence_terms)
        score=overlap+min(len(sentence),220)/1000
        if score>best_score:
            best_score=score
            best_sentence=sentence
    return shorten_text(best_sentence,300)

## 11. Generazione controllata della risposta

Questa parte è il cuore del notebook.

Il sistema non restituisce semplicemente la review più simile. Genera una risposta nuova con una struttura naturale, ma la risposta è sempre vincolata alle evidenze recuperate.


In [15]:
#Definiamo parole semplici per stimare orientamento positivo o negativo
#Questa non è sentiment analysis avanzata
#Serve solo a costruire una risposta generata più leggibile e prudente

POSITIVE_WORDS={
    "good","great","excellent","perfect","works","working","love","loved","nice",
    "easy","fast","solid","reliable","recommend","recommended","useful","best",
    "compatible","fits","fit","fine","well","satisfied","happy","quality",
}

NEGATIVE_WORDS={
    "bad","poor","terrible","awful","broken","break","broke","failed","fails",
    "problem","problems","issue","issues","not","no","never","hard","difficult",
    "slow","weak","worse","worst","return","returned","disappointed","defective",
}

def classify_question(question):
    q=question.lower().strip()
    if any(word in q for word in ["price","cost","how much","expensive","cheap"]):
        return "price"
    if any(word in q for word in ["rating","stars","review score","average rating"]):
        return "rating"
    if any(word in q for word in ["category","type of product","what kind"]):
        return "category"
    if any(word in q for word in ["compatible","work with","works with","fit","fits","support","supports"]):
        return "compatibility"
    if any(word in q for word in ["battery","charge","charging","last long","duration"]):
        return "battery"
    if any(word in q for word in ["quality","good","worth","recommend","reliable","durable"]):
        return "opinion"
    if re.match(r"^(is|are|do|does|did|can|could|will|would|has|have)\b",q):
        return "yes_no"
    return "open"

def sentiment_from_text(text,rating=np.nan):
    text=clean_text(text).lower()
    words=set(re.findall(r"[a-zA-Z]+",text))
    pos=len(words & POSITIVE_WORDS)
    neg=len(words & NEGATIVE_WORDS)

    try:
        rating_value=float(rating)
    except Exception:
        rating_value=np.nan

    if not np.isnan(rating_value):
        if rating_value>=4:
            pos+=2
        elif rating_value<=2:
            neg+=2

    if pos>neg:
        return "positive"
    if neg>pos:
        return "negative"
    return "neutral"


In [16]:
#Risposte dirette dai metadati quando la domanda lo consente
#Queste risposte sono più affidabili delle risposte generate da review
#Se un attributo non è disponibile, la funzione passa al retrieval testuale

def metadata_answer(parent_asin,question):
    parent_asin=str(parent_asin)
    qtype=classify_question(question)
    row=items_by_asin.get(parent_asin)
    if row is None:
        return None

    title=product_title(parent_asin)

    if qtype=="price":
        for col in ["price","price_text"]:
            value=clean_text(row.get(col,"")) if col in row else ""
            if value:
                return {
                    "answer":"For product '"+title+"', the available metadata reports price information as: "+value+".",
                    "source":"metadata",
                    "confidence":"high",
                    "question_type":qtype,
                }

    if qtype=="rating":
        rating=clean_text(row.get("average_rating","")) if "average_rating" in row else ""
        rating_number=clean_text(row.get("rating_number","")) if "rating_number" in row else ""
        if rating:
            extra=(" based on "+rating_number+" ratings") if rating_number else ""
            return {
                "answer":"For product '"+title+"', the metadata reports an average rating of "+rating+extra+".",
                "source":"metadata",
                "confidence":"high",
                "question_type":qtype,
            }

    if qtype=="category":
        for col in ["main_category","categories"]:
            value=clean_text(row.get(col,"")) if col in row else ""
            if value:
                return {
                    "answer":"The product '"+title+"' is associated with this category information: "+shorten_text(value,300)+".",
                    "source":"metadata",
                    "confidence":"high",
                    "question_type":qtype,
                }

    return None

In [17]:
#Generiamo una risposta nuova partendo dalle evidenze recuperate
#La risposta è prudente e cita il numero di evidenze usate
#Le evidenze vengono restituite separatamente per verificare la risposta

def generate_grounded_answer(parent_asin,question,evidence_df):
    parent_asin=str(parent_asin)
    title=product_title(parent_asin)
    qtype=classify_question(question)

    if evidence_df.empty:
        return {
            "answer":"I do not have enough information in the available product data to answer this question reliably.",
            "source":"no_evidence",
            "confidence":"low",
            "question_type":qtype,
        }

    evidence_df=evidence_df[evidence_df["score"]>=MIN_EVIDENCE_SCORE].copy()
    if evidence_df.empty:
        return {
            "answer":"I found some product data, but it is not similar enough to the question to generate a reliable answer.",
            "source":"weak_evidence",
            "confidence":"low",
            "question_type":qtype,
        }

    evidence_df=evidence_df.head(TOP_EVIDENCE).copy()
    evidence_df["sentence"]=evidence_df["text"].apply(lambda text:best_sentence_for_question(text,question))
    evidence_df["sentiment"]=evidence_df.apply(lambda row:sentiment_from_text(row["text"],row.get("rating",np.nan)),axis=1)

    positive_weight=float(evidence_df.loc[evidence_df["sentiment"]=="positive","score"].sum())
    negative_weight=float(evidence_df.loc[evidence_df["sentiment"]=="negative","score"].sum())
    neutral_weight=float(evidence_df.loc[evidence_df["sentiment"]=="neutral","score"].sum())
    best_score=float(evidence_df["score"].max())

    source_counts=evidence_df["evidence_type"].value_counts().to_dict()
    source_text=", ".join([str(k)+": "+str(v) for k,v in source_counts.items()])

    top_sentences=[]
    for sentence in evidence_df["sentence"].tolist():
        sentence=shorten_text(sentence,220)
        if sentence and sentence not in top_sentences:
            top_sentences.append(sentence)
        if len(top_sentences)>=3:
            break

    if best_score>=0.25:
        confidence="medium-high"
    elif best_score>=0.14:
        confidence="medium"
    else:
        confidence="low"

    if qtype in ["yes_no","compatibility","battery","opinion"]:
        if positive_weight>negative_weight*1.25 and positive_weight>0:
            stance="yes"
            opening="Based on the retrieved product evidence, the answer appears to be mostly positive."
        elif negative_weight>positive_weight*1.25 and negative_weight>0:
            stance="no_or_concerns"
            opening="Based on the retrieved product evidence, there are relevant concerns or negative signals."
        else:
            stance="mixed"
            opening="The retrieved evidence is mixed, so I cannot give a clear yes/no answer."
    else:
        stance="informative"
        opening="The most relevant product evidence suggests the following."

    evidence_part=" ".join(["Evidence "+str(i+1)+": "+s for i,s in enumerate(top_sentences)])

    if qtype in ["yes_no","compatibility","battery","opinion"]:
        answer=opening+" For product '"+title+"', I found "+str(len(evidence_df))+" relevant evidence snippets ("+source_text+"). "+evidence_part
    else:
        answer=opening+" For product '"+title+"', I found "+str(len(evidence_df))+" relevant evidence snippets ("+source_text+"). "+evidence_part

    if confidence=="low":
        answer=answer+" Because the similarity score is low, this answer should be treated as tentative."

    return {
        "answer":answer,
        "source":"generated_from_evidence",
        "confidence":confidence,
        "question_type":qtype,
        "stance":stance,
        "best_score":best_score,
        "positive_weight":positive_weight,
        "negative_weight":negative_weight,
        "neutral_weight":neutral_weight,
        "num_evidence":int(len(evidence_df)),
    }

In [18]:
#Funzione finale da usare nel notebook
#Prima prova a rispondere dai metadati per domande semplici
#Poi recupera evidenze e genera una risposta controllata

def answer_product_question(parent_asin,question,top_k=TOP_EVIDENCE):
    parent_asin=str(parent_asin)
    question=clean_text(question)

    direct_answer=metadata_answer(parent_asin,question)
    if direct_answer is not None:
        empty_df=pd.DataFrame(columns=["evidence_type","score","text","rating"])
        return direct_answer,empty_df

    evidence_df=retrieve_evidence(parent_asin,question,top_k=top_k)
    response=generate_grounded_answer(parent_asin,question,evidence_df)

    if not evidence_df.empty:
        evidence_df=evidence_df.copy()
        evidence_df["sentence"]=evidence_df["text"].apply(lambda text:best_sentence_for_question(text,question))
        evidence_df["text_short"]=evidence_df["text"].apply(lambda text:shorten_text(text,500))
        show_cols=["evidence_type","score","rating","question","answer","sentence","text_short"]
        show_cols=[col for col in show_cols if col in evidence_df.columns]
        evidence_df=evidence_df[show_cols]

    return response,evidence_df

## 12. Demo automatica

La cella seguente sceglie automaticamente un prodotto con molte evidenze testuali e prova alcune domande.


In [19]:
#Scegliamo automaticamente un prodotto con molte recensioni o QA
#Questo rende la demo più stabile rispetto a un prodotto casuale
#Puoi sostituire DEMO_PARENT_ASIN con un prodotto specifico

product_evidence_counts=evidence_corpus_df.groupby("parent_asin").size().sort_values(ascending=False)
DEMO_PARENT_ASIN=str(product_evidence_counts.index[0])

print("Demo parent_asin:",DEMO_PARENT_ASIN)
print("Titolo:",product_title(DEMO_PARENT_ASIN))
print("Evidenze disponibili:",int(product_evidence_counts.iloc[0]))

Demo parent_asin: B0043M668G
Titolo: Jabra CRUISER 2 Bluetooth In-Car Speakerphone (Discontinued by Manufacturer)
Evidenze disponibili: 71


In [20]:
#Eseguiamo alcune domande dimostrative sul prodotto scelto
#La risposta viene generata localmente a partire dalle evidenze del prodotto
#La tabella mostra le evidenze che hanno supportato la risposta

demo_questions=[
    "Is it good?",
    "Does it work well?",
    "Is it easy to use?",
    "What do customers complain about?",
    "What is the average rating?",
]

for question in demo_questions:
    print("="*100)
    print("QUESTION:",question)
    response,evidence_df=answer_product_question(DEMO_PARENT_ASIN,question,top_k=TOP_EVIDENCE)
    display(response)
    display(evidence_df.head(5))

QUESTION: Is it good?


{'answer': "Based on the retrieved product evidence, the answer appears to be mostly positive. For product 'Jabra CRUISER 2 Bluetooth In-Car Speakerphone (Discontinued by Manufacturer)', I found 7 relevant evidence snippets (review: 4, qa: 3). Evidence 1: I have one that was gifted to me and I love it....good product. Evidence 2: Review text: Overall this is a a good product and i would recommend it with the following exceptions: The low battery light comes on for a very short time (two hours standby), so twice i have got in the truck and the... Evidence 3: Review text: Best on the market...I have one and it is so good that I bought one for my daughter! Because the similarity score is low, this answer should be treated as tentative.",
 'source': 'generated_from_evidence',
 'confidence': 'low',
 'question_type': 'opinion',
 'stance': 'yes',
 'best_score': 0.12965120332018973,
 'positive_weight': 0.7272388153635879,
 'negative_weight': 0.0,
 'neutral_weight': 0.0,
 'num_evidence': 7}

,evidence_type,score,rating,question,answer,sentence,text_short
0,review,0.129651,4.0,,,I have one that was gifted to me and I love it...,Review title: madear. Review text: I bought th...
1,review,0.116219,4.0,,,Review text: Overall this is a a good product ...,Review title: not bad at all. Review text: Ove...
2,review,0.104205,5.0,,,Review text: Best on the market...I have one a...,Review title: Five Stars. Review text: Best on...
3,review,0.102470,4.0,,,Review text: Works as good as I expected but c...,Review title: Works well. Review text: Works a...
4,qa,0.091565,NaN,iPhone,"I love it, it synchs to my Bluetooth fast when...","Clarity is good, holds a charge long!","Question: iPhone Answer: I love it, it synchs ..."


QUESTION: Does it work well?


{'answer': "Based on the retrieved product evidence, the answer appears to be mostly positive. For product 'Jabra CRUISER 2 Bluetooth In-Car Speakerphone (Discontinued by Manufacturer)', I found 6 relevant evidence snippets (qa: 6). Evidence 1: Question: does it work with iPhone 5 Answer: works with any Bluetooth phone Evidence 2: Question: does it work for all phones Answer: works amazing for all phones with bluetooth connectivity.",
 'source': 'generated_from_evidence',
 'confidence': 'medium-high',
 'question_type': 'yes_no',
 'stance': 'yes',
 'best_score': 0.2865341078122146,
 'positive_weight': 1.2681760929936279,
 'negative_weight': 0.0,
 'neutral_weight': 0.0,
 'num_evidence': 6}

,evidence_type,score,rating,question,answer,sentence,text_short
0,qa,0.286534,NaN,does it work with iPhone 5,works with any Bluetooth phone,Question: does it work with iPhone 5 Answer: w...,Question: does it work with iPhone 5 Answer: w...
1,qa,0.286534,NaN,does it work with iPhone 5,works with any Bluetooth phone,Question: does it work with iPhone 5 Answer: w...,Question: does it work with iPhone 5 Answer: w...
2,qa,0.286534,NaN,does it work with iPhone 5,works with any Bluetooth phone,Question: does it work with iPhone 5 Answer: w...,Question: does it work with iPhone 5 Answer: w...
3,qa,0.136191,NaN,does it work for all phones,works amazing for all phones with bluetooth co...,Question: does it work for all phones Answer: ...,Question: does it work for all phones Answer: ...
4,qa,0.136191,NaN,does it work for all phones,works amazing for all phones with bluetooth co...,Question: does it work for all phones Answer: ...,Question: does it work for all phones Answer: ...


... (5 outputs left out)

,evidence_id,parent_asin,evidence_type,text,question,answer,rating,timestamp,verified_purchase,score


QUESTION: What is the average rating?


{'answer': "The most relevant product evidence suggests the following. For product 'Jabra CRUISER 2 Bluetooth In-Car Speakerphone (Discontinued by Manufacturer)', I found 1 relevant evidence snippets (review: 1). Evidence 1: Review text: Have an older car without an aux jack on my radio so I got this to run Spotify from my phone thru the car speakers. Because the similarity score is low, this answer should be treated as tentative.",
 'source': 'generated_from_evidence',
 'confidence': 'low',
 'question_type': 'rating',
 'stance': 'informative',
 'best_score': 0.09153107600987956,
 'positive_weight': 0.09153107600987956,
 'negative_weight': 0.0,
 'neutral_weight': 0.0,
 'num_evidence': 1}

,evidence_type,score,rating,question,answer,sentence,text_short
0,review,0.091531,4.0,,,Review text: Have an older car without an aux ...,Review title: Great for music... average speak...
1,review,0.066096,3.0,,,"The positives: it was easy to set up, is a sli...",Review title: Just Average. Review text: Havin...


## 13. Cella manuale

Modifica `PARENT_ASIN` e `QUESTION` per interrogare il sistema. Ricorda che l'ASIN è una stringa, quindi va tra virgolette.


In [31]:
#Modifica questi valori per fare domande a mano
#PARENT_ASIN deve essere una stringa tra virgolette
#QUESTION è la domanda utente sul prodotto

PARENT_ASIN="B004M5OYRY"
QUESTION="Is it good with samsung products?"

response,evidence_df=answer_product_question(PARENT_ASIN,QUESTION,top_k=TOP_EVIDENCE)

display(response)
display(evidence_df)

{'answer': 'I found some product data, but it is not similar enough to the question to generate a reliable answer.',
 'source': 'weak_evidence',
 'confidence': 'low',
 'question_type': 'opinion'}

,evidence_type,score,rating,question,answer,sentence,text_short
0,qa,0.049713,NaN,Does this protection cover refurbished Samsung...,"Yes. I should say yes, sometimes. When you mak...",Question: Does this protection cover refurbish...,Question: Does this protection cover refurbish...
1,review,0.040906,5.0,,,Review text: I've used them before with no pro...,Review title: Good deal.. Review text: I've us...
2,review,0.026981,5.0,,,"I got it for my new iPad, and it is much bette...",Review title: Good deal. Review text: This is ...
3,review,0.015871,5.0,,,Review text: APPEARS TO BE A GOOD COMPANY BUT ...,Review title: ??. Review text: APPEARS TO BE A...
4,review,0.005578,5.0,,,"iPhones get dropped, cats knock the tablet ont...",Review title: Accidental Hardware and Service ...


## 14. Sanity check quantitativo

Questa valutazione non misura la qualità linguistica finale della risposta generata. Misura solo se la fase di recupero riesce a ritrovare, per domande QA già presenti, evidenze dello stesso prodotto.

È un controllo tecnico utile: se il retrieval funziona male, anche la generazione finale sarà debole.


In [22]:
#Valutiamo il retrieval su un campione di domande QA storiche
#Per ogni domanda, controlliamo se tra le evidenze recuperate compare una QA simile
#Questa non è una valutazione completa del QA generativo

def run_retrieval_sanity_check(sample_size=200,top_k=5):
    if qa_evidence_df.empty:
        return pd.DataFrame(),{}

    sample_df=qa_evidence_df.sample(n=min(sample_size,len(qa_evidence_df)),random_state=RANDOM_STATE).reset_index(drop=True)
    rows=[]

    for row in tqdm(sample_df.itertuples(index=False),total=len(sample_df),desc="Sanity check QA"):
        question=getattr(row,"question")
        parent_asin=getattr(row,"parent_asin")
        retrieved_df=retrieve_evidence(parent_asin,question,top_k=top_k)
        hit_same_qa=False
        if not retrieved_df.empty:
            qa_matches=retrieved_df[retrieved_df["evidence_type"]=="qa"]
            hit_same_qa=any(qa_matches["question"].fillna("").astype(str)==str(question))
        rows.append({
            "parent_asin":parent_asin,
            "question":question,
            "hit_same_qa_at_k":bool(hit_same_qa),
            "retrieved_count":int(len(retrieved_df)),
            "best_score":float(retrieved_df["score"].max()) if not retrieved_df.empty else 0.0,
        })

    result_df=pd.DataFrame(rows)
    summary={
        "sample_size":int(len(result_df)),
        "top_k":int(top_k),
        "hit_same_qa_at_k":float(result_df["hit_same_qa_at_k"].mean()) if len(result_df) else np.nan,
        "mean_best_score":float(result_df["best_score"].mean()) if len(result_df) else np.nan,
    }
    return result_df,summary

sanity_df,sanity_summary=run_retrieval_sanity_check(sample_size=200,top_k=5)
display(sanity_summary)
display(sanity_df.head(10))

Sanity check QA: 100%|██████████| 200/200 [00:02<00:00, 78.03it/s]


{'sample_size': 200,
 'top_k': 5,
 'hit_same_qa_at_k': 1.0,
 'mean_best_score': 0.610674979519913}

,parent_asin,question,hit_same_qa_at_k,retrieved_count,best_score
0,B00856XE5E,Is this model Netflix streaming compatible/ready?,True,2,0.399923
1,B00DM5F19A,Will this unit receive am radio .? Is antenna ...,True,5,0.595149
2,B001R84DIY,will this work in dell Vostro 1000?,True,5,0.428825
3,B008JWK664,Does it have Night Vision?,True,5,0.750864
4,B004M5OYRY,Can I buy a plan even though I've had my ipad ...,True,5,0.802887
5,B00FRMAOIO,"I have a non-wireless printer, can I link this...",True,5,0.837582
6,B00CU9GKTO,will this laptop work well with sims?,True,5,0.423330
7,B00EZBS38Q,Does it work with the 2013 edition?,True,5,0.491688
8,B00B1HXHLU,Does this printer have a PC fax function? I ne...,True,5,0.910754
9,B00DILTY0U,I'm a newbie to this. How do you add more memory?,True,4,0.657641


## 15. Salvataggio riepilogo

In [23]:
#Salviamo un riepilogo del notebook e degli artefatti prodotti
#Questo file può essere citato quando descriviamo il lavoro svolto
#Le metriche del sanity check vanno interpretate con cautela

summary={
    "dataset":DATASET_NAME,
    "items":int(len(items_df)),
    "qa_rows":int(len(qa_df)),
    "reviews_loaded":int(len(reviews_df)),
    "evidence_rows":int(len(evidence_corpus_df)),
    "evidence_products":int(evidence_corpus_df["parent_asin"].nunique()),
    "evidence_type_counts":{str(k):int(v) for k,v in evidence_corpus_df["evidence_type"].value_counts().to_dict().items()},
    "tfidf_shape":[int(evidence_matrix.shape[0]),int(evidence_matrix.shape[1])],
    "sanity_summary":sanity_summary,
    "artifacts":{
        "evidence_corpus":str(EVIDENCE_CORPUS_PATH),
        "vectorizer":str(VECTORIZER_PATH),
        "matrix":str(MATRIX_PATH),
    },
}

SUMMARY_PATH.write_text(json.dumps(summary,indent=2),encoding="utf-8")

display(summary)
print("Riepilogo salvato in:",SUMMARY_PATH)

{'dataset': 'test_dataset2',
 'items': 19000,
 'qa_rows': 166191,
 'reviews_loaded': 250000,
 'evidence_rows': 435191,
 'evidence_products': 19000,
 'evidence_type_counts': {'review': 250000, 'qa': 166191, 'metadata': 19000},
 'tfidf_shape': [435191, 120000],
 'sanity_summary': {'sample_size': 200,
  'top_k': 5,
  'hit_same_qa_at_k': 1.0,
  'mean_best_score': 0.610674979519913},
 'artifacts': {'evidence_corpus': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\data\\processed\\test_dataset2_recommender\\notebook06_generative_qa\\qa_evidence_corpus.parquet',
  'vectorizer': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\data\\processed\\test_dataset2_recommender\\notebook06_generative_qa\\qa_evidence_tfidf_vectorizer.joblib',
  'matrix': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\data\\processed\\test_dataset2_recommender\\notebook06_generative_qa\\qa_evidence_tfidf_matrix.npz'}}

Riepilogo salvato in: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset2_recommender\notebook06_generative_qa\notebook06_generative_qa_summary.json


## 16. Conclusione

Questo notebook implementa una versione leggera di **generative product-aware QA**:

- usa attributi, QA e recensioni come fonti di conoscenza;
- recupera le evidenze più rilevanti per il prodotto e per la domanda;
- genera una risposta nuova e leggibile tramite regole controllate;
- mostra sempre le evidenze usate;
- evita di rispondere quando non ci sono informazioni sufficienti.

Il limite principale è che non usa un vero modello neurale generativo addestrato sul task. Per una tesi triennale, però, questa versione è più difendibile: è trasparente, riproducibile, non usa API esterne e permette di spiegare chiaramente il collegamento tra domanda, evidenze e risposta.
